# 21 — Cost, Latency, and Token Engineering

    ## Scenario and success criteria

    A support answer must retain a termination clause while reducing irrelevant context.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Trace quality, tokens, latency, and cost together.
- Identify dominated configurations.
- Refuse optimizations that cross a quality gate.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 21 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

Removing the relevant clause makes a policy cheap and fast but unusable.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab21 import dominates, quality_gate, run_policy

boilerplate = "General terms. " * 80
clause = "The company may terminate with 30 days notice."
full = run_policy("full", boilerplate + clause, "May the company terminate?")
pruned = run_policy("pruned", clause, "May the company terminate?")
lossy = run_policy("lossy", boilerplate, "May the company terminate?")

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
for result in (full, pruned, lossy):
    print(result, "quality", quality_gate(result, "yes"))
print("pruned dominates full", dominates(pruned, full, expected="yes"))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert quality_gate(full, "yes") and quality_gate(pruned, "yes")
assert not quality_gate(lossy, "yes")
assert dominates(pruned, full, expected="yes")
assert not dominates(lossy, full, expected="yes")

## Production upgrade

Optimize in this order: remove unnecessary work, route by measured need, bound outputs and retries, cache only within authorization scope, then consider model changes. Report percentiles and quality-adjusted cost by slice.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.